In [3]:
#
import urllib
from bs4 import BeautifulSoup
import requests
import numpy
import os
import re
import time
import json
import datetime
import unicodedata

In [5]:
url_general = "https://maldita.es"
html_general = requests.get(url_general).text
soup_general = BeautifulSoup(html_general, 'html.parser')

In [7]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

base_url = "https://maldita.es"
bulo_url = "/malditobulo/"
response = requests.get("https://maldita.es/malditobulo/")#tambien aca podemos poner .text, y en la siguiente linea no lo ponemos
soup = BeautifulSoup(response.text, "html.parser")
ultimo_button = soup.find("button", string=">>")
numero_paginas = 0
if ultimo_button:
  pagina_url = ultimo_button["data-page-url"]
  numero_paginas = pagina_url.strip("/").split("/")[-1]
  print(numero_paginas)

563


In [8]:
import pandas as pd
import urllib
import re
import datetime
numero_paginas = 1
articulos_data = []
for i in range(1, int(numero_paginas) + 1):
  contador_articulos_pagina = 0
  bulo = urljoin(base_url, bulo_url + str(i))
  response_bulo = requests.get(bulo)
  soup_bulo = BeautifulSoup(response_bulo.text, 'html.parser')
  main_articulos_contenedor = soup_bulo.find('div', class_='grid grid-cols-1 md:grid-cols-2 lg:grid-cols-3 gap-4')
  #print(main_articulos_contenedor)
  if main_articulos_contenedor:
    articulos_links = main_articulos_contenedor.find_all('a', href=re.compile(r'/malditobulo/\d+'))
    #print(articulos_links)
    for articulo_link in articulos_links:
      contador_articulos_pagina+=1
      titulo = 'N/A'
      url = 'N/A'
      fecha = 'N/A'
      categoria = 'N/A'
      texto = 'N/A'

      url = urllib.parse.urljoin(url_general, articulo_link['href'])

      titulo_div_elemento = articulo_link.find('div', class_='bg-white rounded-t-2x1 rounded-b1-2x1 p-4 font-semibold text-lg mx-6 mb-4 -mt-12 lead:')
      if titulo_div_elemento:
        titulo = titulo_div_elemento.get_text(strip=True)

      fecha_match = re.search(r'/malditobulo/(\d{8})', url)
      fecha_str = fecha_match.group(1)

      categoria_div_elemento = articulo_link.find('div', class_='flex items-center gap-2 px-2 py-1 rounded-lg')
      if categoria_div_elemento:
        catgeoria = categoria_div_elemento.get_text(strip=True)

      try:
        fecha = datetime.datetime.strptime(fecha_str, '%Y%m%d').strftime('%Y-%m-%d')
      except ValueError:
        fecha = fecha_str
      response_articulo = requests.get(url,allow_redirects=False)
      soup_articulo = BeautifulSoup(response_articulo.text, 'html.parser')
      contenido_div_elemento = soup_articulo.find('div', class_='w-full md:w-[657px] mx-auto')
      contenido_texto = contenido_div_elemento.find('span')
      articulos_data.append({'titulo': titulo, 'url': url, 'fecha': fecha, 'categoria': categoria, 'texto': texto})
  else:
    print(f"Error: Contenedor de Articulos principal con class 'grid grid-cols-1 md:grid-cols-2 lg:grid-cols-3 gap-4' no fueron econtrados")
  print(f"Se econtraron {contador_articulos_pagina} articulos en la pagina {i}.")
df_articulos = pd.DataFrame(articulos_data)
print(f"Se encontraron {len(df_articulos)} articulos en total de {numero_paginas} Paginas con Web Scrapping.")
output_file = "data.json"
#json_data = df_articulos.to_json(orient='records',lines=True,date_format="iso",default_handler=str)
df_articulos.to_json(output_file,orient='records',lines=True,date_format="iso",default_handler=str)
print(f"\nJSON guardado en {output_file}")

Se econtraron 18 articulos en la pagina 1.
Se encontraron 18 articulos en total de 1 Paginas con Web Scrapping.

JSON guardado en data.json
